# 🤖 Sistema Multiagente de Atención al Cliente con CrewAI

Este notebook implementa un sistema completo de atención al cliente usando **CrewAI**.  
Incluye 5 agentes especializados, datos sintéticos y simulaciones de tickets reales.


**Stack:** `crewai` · `anthropic` · `faker` · `rich`


## 🏛️ Arquitectura del Sistema Multiagente

Este sistema implementa una arquitectura **secuencial dinámica** utilizando el framework **CrewAI**. El objetivo es simular un departamento de atención al cliente moderno donde los tickets entrantes son primero evaluados y luego derivados al especialista adecuado.

### 👥 Los Agentes

El sistema ("Crew") está compuesto por 5 agentes de IA, cada uno con un `role` (rol), `goal` (objetivo) y `backstory` (historia de fondo) que define su personalidad y comportamiento:

1.  **🔀 Agente de Triaje y Clasificación:** Actúa como el primer punto de contacto. Su única función es analizar el ticket entrante, determinar su naturaleza (devolución, soporte, queja, consulta), asignar una prioridad y enviarlo al siguiente nivel. *Solo tiene acceso a la herramienta de consultar cliente.*
2.  **📦 Especialista en Devoluciones:** Se encarga exclusivamente de los tickets clasificados como "devolucion". *Tiene acceso a herramientas para consultar pedidos, verificar la elegibilidad de devoluciones y generar etiquetas de envío.*
3.  **🔧 Especialista en Soporte Técnico:** Atiende problemas de funcionamiento de productos ("soporte_tecnico"). *Tiene acceso a la base de conocimiento (KB) y al sistema de incidencias activas.*
4.  **💬 Especialista en Consultas:** Gestiona preguntas generales sobre envíos o políticas, y también se encarga de apaciguar a los clientes en caso de "queja_grave". *Tiene acceso a pedidos, clientes e incidencias.*
5.  **✅ Agente de Control de Calidad (QA):** El último paso del flujo. Revisa la respuesta generada por el especialista evaluando 4 dimensiones (Precisión, Política, Tono y Completitud). Si la respuesta no es adecuada para el segmento del cliente (ej. un cliente VIP), la rechaza o la reescribe antes de que llegue al usuario final.

### 🛠️ Herramientas (Tools)

Para que los agentes no inventen datos (alucinaciones), se les ha dotado de herramientas personalizadas (basadas en `BaseTool` de LangChain/CrewAI) que conectan con "sistemas externos" (en este caso, diccionarios de datos sintéticos):

*   `consultar_cliente`: Busca información del CRM (segmento, historial).
*   `consultar_pedido`: Busca información del ERP (estado de envío, fechas).
*   `buscar_knowledge_base`: Busca manuales de solución de problemas.
*   `verificar_incidencias`: Consulta si hay caídas de servicio o problemas logísticos globales.
*   `verificar_devolucion` y `crear_etiqueta_devolucion`: Interactúan con el sistema de logística inversa.

### 🔄 Flujo de Ejecución (Pipeline)

Para cada ticket, el proceso (Process) es el siguiente:
1.  **Generación de Contexto:** Se inyecta la información básica del ticket y del cliente.
2.  **Tarea 1 (Triaje):** El agente de triaje clasifica el ticket y genera un JSON estructurado.
3.  **Selección Dinámica:** Basado en la clasificación de la Tarea 1, el código Python selecciona qué agente especialista intervendrá y genera una **Tarea 2** personalizada.
4.  **Tarea 2 (Resolución):** El especialista seleccionado usa sus herramientas para investigar y redactar un borrador de respuesta.
5.  **Tarea 3 (QA):** El agente de QA evalúa el borrador frente al mensaje original del cliente y las políticas de la empresa, emitiendo la respuesta final aprobada.

### 📂 Estructura del Notebook

1.  **Instalación e Imports:** Configuración del entorno y conexión con el LLM (OpenAI `gpt-4o-mini` o `crewai.LLM`).
2.  **Datos Sintéticos:** Generación aleatoria y realista de clientes, pedidos, catálogo de productos y una base de conocimiento técnica usando la librería `Faker`.
3.  **Tools (Herramientas):** Definición de las clases `BaseTool` con esquemas Pydantic para tipado estricto.
4.  **Definición de Agentes y Tareas:** Creación de las instancias de `Agent` y la función `construir_crew` que orquesta las `Task`.
5.  **Ejecución y Resultados:** Bucle principal que pasa tickets de muestra por el sistema y un apartado para probar con mensajes personalizados.


## 1. Instalación de dependencias

In [1]:
# ⚠️  Ejecutar una sola vez. Reiniciar el kernel después si es necesario.
%pip install crewai crewai-tools openai langchain-openai faker rich langchain-core python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.54.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.2 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports y configuración

In [2]:
import os
import json
import random
from datetime import datetime, timedelta
from typing import Optional

from faker import Faker
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import print as rprint

from crewai import Agent, Task, Crew, Process, LLM
# Importación robusta para BaseTool
try:
    from crewai.tools import BaseTool
except ImportError:
    from langchain_core.tools import BaseTool

from langchain_openai import ChatOpenAI

# ── Configuración ──────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️ WARNING: OPENAI_API_KEY no encontrada en el entorno o archivo .env")

console = Console()
fake = Faker("es_ES")
random.seed(42)

llm = LLM(
    model="gpt-4o-mini",
    temperature=0.2,
    api_key=os.environ.get("OPENAI_API_KEY"),
)

console.print("[bold green]✅ Entorno y LLM reiniciados con la nueva clave[/bold green]")

✅ Entorno y LLM reiniciados con la nueva clave

## 3. Datos sintéticos

Generamos una base de datos simulada con:
- **Clientes** con historial de compras y segmento
- **Pedidos** con estados y productos
- **Base de conocimiento** para soporte técnico
- **Tickets de soporte** representativos de cada categoría


In [3]:
# ── Catálogo de productos ──────────────────────────────────────────────────────
PRODUCTOS = [
    {"id": "P001", "nombre": "SmartWatch Pro X2",      "categoria": "electronica", "precio": 249.99},
    {"id": "P002", "nombre": "Auriculares BT NoiseX",   "categoria": "electronica", "precio": 89.99},
    {"id": "P003", "nombre": "Tablet FlexyPad 10",      "categoria": "electronica", "precio": 349.99},
    {"id": "P004", "nombre": "Zapatillas RunFast V3",   "categoria": "deporte",     "precio": 119.99},
    {"id": "P005", "nombre": "Mochila TrailMaster 40L", "categoria": "deporte",     "precio": 74.99},
    {"id": "P006", "nombre": "Cafetera EspressoMax",    "categoria": "hogar",       "precio": 159.99},
    {"id": "P007", "nombre": "Robot Aspirador CleanBot","categoria": "hogar",       "precio": 299.99},
]

SEGMENTOS = ["estandar", "premium", "vip"]
ESTADOS_PEDIDO = ["entregado", "en_transito", "preparando", "devuelto", "cancelado"]

# ── Generación de clientes ─────────────────────────────────────────────────────
def generar_clientes(n=50):
    clientes = {}
    for _ in range(n):
        cid = f"C{fake.numerify('####')}"
        clientes[cid] = {
            "id": cid,
            "nombre": fake.name(),
            "email": fake.email(),
            "telefono": fake.phone_number(),
            "segmento": random.choices(SEGMENTOS, weights=[60, 30, 10])[0],
            "fecha_registro": fake.date_between(start_date="-3y", end_date="-1m").isoformat(),
            "total_pedidos": random.randint(1, 25),
            "nps_historico": random.randint(3, 10),
        }
    return clientes

# ── Generación de pedidos ──────────────────────────────────────────────────────
def generar_pedidos(clientes, n=120):
    pedidos = {}
    for _ in range(n):
        oid = f"ORD-{fake.numerify('######')}"
        producto = random.choice(PRODUCTOS)
        cliente = random.choice(list(clientes.values()))
        dias_atras = random.randint(1, 90)
        fecha_compra = (datetime.now() - timedelta(days=dias_atras)).date()
        fecha_entrega = (fecha_compra + timedelta(days=random.randint(2, 7))) if dias_atras > 5 else None
        pedidos[oid] = {
            "id": oid,
            "cliente_id": cliente["id"],
            "producto": producto,
            "cantidad": random.randint(1, 3),
            "total": round(producto["precio"] * random.randint(1, 3), 2),
            "estado": random.choices(ESTADOS_PEDIDO, weights=[50, 20, 15, 10, 5])[0],
            "fecha_compra": fecha_compra.isoformat(),
            "fecha_entrega": fecha_entrega.isoformat() if fecha_entrega else None,
            "dias_desde_compra": dias_atras,
            "numero_seguimiento": fake.bothify("ES##########ES"),
        }
    return pedidos

# ── Base de conocimiento técnico ───────────────────────────────────────────────
KNOWLEDGE_BASE = {
    "SmartWatch Pro X2": [
        "Para sincronizar con iPhone: Ajustes → Bluetooth → Buscar 'SmartWatch Pro X2' → Emparejar.",
        "Si la batería no carga: Limpiar los contactos del cargador con un paño seco. Asegurarse de usar el cargador oficial.",
        "Reseteo de fábrica: Mantener botón lateral 10 segundos hasta ver logo. Todos los datos se borran.",
        "Actualizar firmware: App SmartSync → Mi dispositivo → Actualizar. Requiere batería >30%.",
    ],
    "Auriculares BT NoiseX": [
        "Para el modo de cancelación de ruido: Doble toque en orejera izquierda. LED azul = ANC activo.",
        "Emparejar con nuevo dispositivo: Con auriculares apagados, mantener botón 5s hasta LED parpadeante.",
        "Sin sonido en una orejera: Resetear tocando ambas orejeras 3 veces seguidas.",
        "Carga inalámbrica: Compatible con cargadores Qi. Colocar en el centro del pad de carga.",
    ],
    "Robot Aspirador CleanBot": [
        "Configurar horario de limpieza: App CleanHome → Programar → Seleccionar días y hora.",
        "El robot no regresa a la base: Verificar que la base esté en zona libre de obstáculos 1.5m a cada lado.",
        "Error E02 (cepillo bloqueado): Retirar el cepillo principal y limpiar pelos/hilos enredados.",
        "Mapa incorrecto: Iniciar limpieza completa en modo cartografía desde la app para remapear.",
    ],
    "general": [
        "Política de devoluciones: 30 días desde la entrega para cualquier producto sin usar.",
        "Productos dañados en transporte: 60 días con foto del daño. Recogida gratuita.",
        "Garantía: 2 años para electrónica, 1 año para productos de deporte y hogar.",
        "Envío estándar: 3-5 días laborables. Express 24h disponible por 4,99€ adicionales.",
        "Métodos de pago: Tarjeta, PayPal, Bizum, financiación 0% en compras >200€.",
    ]
}

# ── Generación de tickets ──────────────────────────────────────────────────────
TICKETS_MUESTRA = [
    {
        "id": "TK-001",
        "categoria": "devolucion",
        "prioridad_esperada": "media",
        "cliente_id": None,   # se asigna dinámicamente
        "pedido_id": None,
        "mensaje": (
            "Hola, compré un SmartWatch Pro X2 hace 3 semanas y la pantalla "
            "ha dejado de funcionar tras una caída. El reloj cayó desde mi mesita "
            "de noche, unos 70cm. ¿Cubre la garantía este tipo de daño? "
            "Si no, ¿puedo devolverlo igualmente?"
        ),
    },
    {
        "id": "TK-002",
        "categoria": "soporte_tecnico",
        "prioridad_esperada": "media",
        "cliente_id": None,
        "pedido_id": None,
        "mensaje": (
            "Buenos días. Mi Robot Aspirador CleanBot lleva 2 días mostrando "
            "el error E02 y no termina ningún ciclo de limpieza. Ya lo reinicié "
            "pero sigue igual. ¿Qué puedo hacer? Necesito que funcione para el "
            "fin de semana que tengo visita."
        ),
    },
    {
        "id": "TK-003",
        "categoria": "consulta",
        "prioridad_esperada": "baja",
        "cliente_id": None,
        "pedido_id": None,
        "mensaje": (
            "Quería saber el estado de mi pedido ORD-123456. "
            "Lo hice hace 4 días y en el email dice 'en tránsito' pero "
            "el número de seguimiento no aparece en la web de Correos. "
            "¿Es normal? ¿Cuándo llega?"
        ),
    },
    {
        "id": "TK-004",
        "categoria": "queja_grave",
        "prioridad_esperada": "alta",
        "cliente_id": None,
        "pedido_id": None,
        "mensaje": (
            "Esto es inaceptable. Llevo 3 semanas intentando devolver unos "
            "auriculares defectuosos y nadie me hace caso. Ya he llamado 4 veces, "
            "mandé 6 emails y abrí 2 tickets que se cerraron solos sin resolver. "
            "Soy cliente desde hace 5 años y jamás me han tratado así. "
            "Si no resuelven esto HOY, haré una reclamación formal y lo publicaré "
            "en redes sociales. Quiero hablar con un responsable."
        ),
    },
    {
        "id": "TK-005",
        "categoria": "devolucion",
        "prioridad_esperada": "baja",
        "cliente_id": None,
        "pedido_id": None,
        "mensaje": (
            "Hola! Recibí la mochila TrailMaster hace una semana, está perfecta, "
            "pero me regalaron la misma y tengo dos iguales. ¿Puedo devolverla "
            "sin problema? No tiene ningún defecto, ni siquiera la he usado. "
            "Gracias 😊"
        ),
    },
]

# ── Instanciar datos ───────────────────────────────────────────────────────────
CLIENTES = generar_clientes(50)
PEDIDOS  = generar_pedidos(CLIENTES, 120)

# ── Asignar cliente real y PEDIDO COHERENTE a cada ticket ─────────────────────

cliente_ids = list(CLIENTES.keys())

# Producto lookup por nombre
PRODUCTOS_POR_NOMBRE = {p["nombre"]: p for p in PRODUCTOS}

def hacer_pedido(product_name, dias_atras, estado="entregado", cantidad=1):
    """Crea un pedido coherente con el mensaje del ticket."""
    from faker import Faker as _Faker
    _fake = _Faker("es_ES")
    producto = PRODUCTOS_POR_NOMBRE[product_name]
    oid = f"ORD-{_fake.numerify('######')}"
    fecha_compra = (datetime.now() - timedelta(days=dias_atras)).date()
    fecha_entrega = (fecha_compra + timedelta(days=4)) if dias_atras > 4 else None
    return oid, {
        "id": oid,
        "cliente_id": None,           # se rellena abajo
        "producto": producto,
        "cantidad": cantidad,
        "total": round(producto["precio"] * cantidad, 2),
        "estado": estado,
        "fecha_compra": fecha_compra.isoformat(),
        "fecha_entrega": fecha_entrega.isoformat() if fecha_entrega else None,
        "dias_desde_compra": dias_atras,
        "numero_seguimiento": _fake.bothify("ES##########ES"),
    }

# Pedidos específicos alineados con cada mensaje de ticket
oid1, ped1 = hacer_pedido("SmartWatch Pro X2",   dias_atras=21, estado="entregado")
oid2, ped2 = hacer_pedido("Robot Aspirador CleanBot", dias_atras=45, estado="entregado")
oid3, ped3 = hacer_pedido("Tablet FlexyPad 10",  dias_atras=4,  estado="en_transito")
oid4, ped4 = hacer_pedido("Auriculares BT NoiseX", dias_atras=28, estado="entregado")
oid5, ped5 = hacer_pedido("Mochila TrailMaster 40L", dias_atras=7, estado="entregado")

# Para TK-003 el cliente mencionó "ORD-123456" — forzamos ese ID
ped3["id"] = "ORD-123456"
oid3 = "ORD-123456"

pedidos_ticket = [
    (oid1, ped1),
    (oid2, ped2),
    (oid3, ped3),
    (oid4, ped4),
    (oid5, ped5),
]

for i, ticket in enumerate(TICKETS_MUESTRA):
    cid = cliente_ids[i % len(cliente_ids)]
    oid, pedido = pedidos_ticket[i]
    pedido["cliente_id"] = cid          # vincular cliente al pedido
    PEDIDOS[oid] = pedido               # añadir al pool global de pedidos

    ticket["cliente_id"] = cid
    ticket["pedido_id"]  = oid
    ticket["cliente"]    = CLIENTES[cid]
    ticket["pedido"]     = pedido


# ── Resumen visual ─────────────────────────────────────────────────────────────
table = Table(title="📦 Datos Sintéticos Generados", show_header=True, header_style="bold cyan")
table.add_column("Entidad",   style="cyan")
table.add_column("Cantidad",  justify="right")
table.add_column("Notas")
table.add_row("Clientes",  str(len(CLIENTES)), "3 segmentos: estándar, premium, VIP")
table.add_row("Pedidos",   str(len(PEDIDOS)),  "5 estados posibles")
table.add_row("Productos", str(len(PRODUCTOS)),"3 categorías: electrónica, deporte, hogar")
table.add_row("KB artículos", str(sum(len(v) for v in KNOWLEDGE_BASE.values())), "4 productos + FAQ general")
table.add_row("Tickets muestra", str(len(TICKETS_MUESTRA)), "5 categorías representadas")
console.print(table)


                      📦 Datos Sintéticos Generados                       
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Entidad         ┃ Cantidad ┃ Notas                                     ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Clientes        │       50 │ 3 segmentos: estándar, premium, VIP       │
│ Pedidos         │      125 │ 5 estados posibles                        │
│ Productos       │        7 │ 3 categorías: electrónica, deporte, hogar │
│ KB artículos    │       17 │ 4 productos + FAQ general                 │
│ Tickets muestra │        5 │ 5 categorías representadas                │
└─────────────────┴──────────┴───────────────────────────────────────────┘

## 4. Herramientas (Tools) de los agentes

Cada agente tiene acceso a herramientas específicas que simulan llamadas
a sistemas externos: base de datos de pedidos, políticas, sistema de reembolsos, etc.


In [4]:
import json
from pydantic import BaseModel, Field

# ── Pydantic schemas for typed tool inputs ────────────────────────────────────

class PedidoInput(BaseModel):
    pedido_id: str = Field(..., description="ID del pedido, ej. ORD-123456")

class ClienteInput(BaseModel):
    cliente_id: str = Field(..., description="ID del cliente, ej. C1234")

class KnowledgeInput(BaseModel):
    query: str = Field(..., description="Nombre del producto o descripción del problema")

class EmptyInput(BaseModel):
    pass

# ── Tool: Verificar Incidencias ───────────────────────────────────────────────
class VerificarIncidenciasTool(BaseTool):
    name: str = "verificar_incidencias"
    description: str = "Comprueba si hay incidencias activas en el sistema. No requiere parámetros."
    args_schema: type[BaseModel] = EmptyInput

    def _run(self, **kwargs) -> str:
        incidencias = [
            {"id": "INC-001", "producto": "general",
             "descripcion": "Retrasos en entregas zona norte por meteorología.", "activa": True},
            {"id": "INC-002", "producto": "SmartWatch Pro X2",
             "descripcion": "Problema de sincronización detectado en iOS 17.", "activa": True},
        ]
        activas = [i for i in incidencias if i["activa"]]
        return json.dumps(activas, indent=2, ensure_ascii=False) if activas else "Sin incidencias."


# ── Tool: Buscar Knowledge Base ───────────────────────────────────────────────
class BuscarKnowledgeTool(BaseTool):
    name: str = "buscar_knowledge_base"
    description: str = (
        "Busca artículos de ayuda técnica. "
        "Parámetro: query — nombre del producto o descripción del problema."
    )
    args_schema: type[BaseModel] = KnowledgeInput

    def _run(self, query: str = "general", **kwargs) -> str:
        q = query.lower()
        resultados = []
        for producto, articulos in KNOWLEDGE_BASE.items():
            if producto.lower() in q or any(w in q for w in producto.lower().split()):
                resultados.extend([f"[{producto}] {a}" for a in articulos])
        if not resultados:
            resultados.extend([f"[general] {a}" for a in KNOWLEDGE_BASE.get("general", [])])
        return "\n".join(resultados[:6]) if resultados else "No se encontraron artículos."


# ── Tool: Consultar Pedido ────────────────────────────────────────────────────
class ConsultarPedidoTool(BaseTool):
    name: str = "consultar_pedido"
    description: str = (
        "Obtiene los detalles de un pedido. "
        "Parámetro: pedido_id — identificador del pedido, ej. ORD-123456."
    )
    args_schema: type[BaseModel] = PedidoInput

    def _run(self, pedido_id: str = "", **kwargs) -> str:
        result = PEDIDOS.get(pedido_id, {"error": f"Pedido '{pedido_id}' no encontrado"})
        return json.dumps(result, indent=2, ensure_ascii=False, default=str)


# ── Tool: Consultar Cliente ───────────────────────────────────────────────────
class ConsultarClienteTool(BaseTool):
    name: str = "consultar_cliente"
    description: str = (
        "Consulta el historial del cliente. "
        "Parámetro: cliente_id — identificador del cliente, ej. C1234."
    )
    args_schema: type[BaseModel] = ClienteInput

    def _run(self, cliente_id: str = "", **kwargs) -> str:
        result = CLIENTES.get(cliente_id, {"error": f"Cliente '{cliente_id}' no encontrado"})
        return json.dumps(result, indent=2, ensure_ascii=False, default=str)


# ── Tool: Verificar Devolución ────────────────────────────────────────────────
class VerificarDevolucionTool(BaseTool):
    name: str = "verificar_devolucion"
    description: str = (
        "Verifica si un pedido es elegible para devolución. "
        "Parámetro: pedido_id — identificador del pedido."
    )
    args_schema: type[BaseModel] = PedidoInput

    def _run(self, pedido_id: str = "", **kwargs) -> str:
        pedido = PEDIDOS.get(pedido_id, {})
        dias = pedido.get("dias_desde_compra", 999)
        estado = pedido.get("estado", "")
        cat = pedido.get("producto", {}).get("categoria", "")

        if estado in ("devuelto", "cancelado"):
            return json.dumps({"elegible": False, "razon": f"Estado '{estado}' no permite devolución."})
        if dias <= 30:
            return json.dumps({"elegible": True,
                "razon": f"Dentro del plazo de 30 días ({dias} días).", "tipo": "estandar"})
        elif dias <= 60:
            return json.dumps({"elegible": True,
                "razon": f"Elegible por daño en transporte ({dias} días). Se requiere foto.", "tipo": "dano"})
        else:
            garantia = 730 if cat == "electronica" else 365
            if dias <= garantia:
                return json.dumps({"elegible": True,
                    "razon": f"En garantía ({dias} días). Requiere revisión técnica.", "tipo": "garantia"})
            return json.dumps({"elegible": False,
                "razon": f"Fuera de plazo de devolución y garantía ({dias} días)."})


# ── Tool: Crear Etiqueta Devolución ──────────────────────────────────────────
class CrearEtiquetaTool(BaseTool):
    name: str = "crear_etiqueta_devolucion"
    description: str = (
        "Genera una etiqueta de devolución prepagada. "
        "Parámetro: pedido_id — identificador del pedido a devolver."
    )
    args_schema: type[BaseModel] = PedidoInput

    def _run(self, pedido_id: str = "", **kwargs) -> str:
        import random, string
        codigo = "RET-" + "".join(random.choices(string.digits, k=4)) + \
                 "-" + "".join(random.choices(string.ascii_uppercase, k=4))
        return json.dumps({
            "etiqueta_id": codigo,
            "pedido_id": pedido_id,
            "transportista": "SEUR",
            "instrucciones": "Lleva el paquete a cualquier oficina SEUR en los próximos 14 días.",
            "reembolso_plazo": "3-5 días hábiles tras recibir el paquete."
        }, ensure_ascii=False)


# ── Instanciar herramientas ───────────────────────────────────────────────────
tool_pedido      = ConsultarPedidoTool()
tool_kb          = BuscarKnowledgeTool()
tool_cliente     = ConsultarClienteTool()
tool_incidencias = VerificarIncidenciasTool()
tool_devolucion  = VerificarDevolucionTool()
tool_etiqueta    = CrearEtiquetaTool()

console.print("[bold green]✅ Herramientas corregidas con args_schema Pydantic[/bold green]")


✅ Herramientas corregidas con args_schema Pydantic

## 5. Definición de agentes

Cada agente tiene un `role`, `goal` y `backstory` específicos que moldean
su comportamiento. Los agentes especialistas tienen acceso solo a las
herramientas que necesitan.


In [5]:
# ── RE-INICIALIZACIÓN DE AGENTES ──────────────────────────────────────────────
# Ejecutando esta celda aseguramos que los agentes usen las herramientas actualizadas

agente_triage = Agent(
    role="Agente de Triaje y Clasificación",
    goal="Clasificar tickets y asignar prioridad. Output siempre JSON.",
    backstory="Experto en clasificación de intenciones.",
    tools=[tool_cliente],
    llm=llm, verbose=True, allow_delegation=False
)

agente_devoluciones = Agent(
    role="Especialista en Devoluciones",
    goal="Resolver solicitudes de devolución y reembolsos.",
    backstory="Experto en políticas de retorno.",
    tools=[tool_pedido, tool_devolucion, tool_etiqueta, tool_cliente],
    llm=llm, verbose=True, allow_delegation=False
)

agente_soporte = Agent(
    role="Especialista en Soporte Técnico",
    goal="Diagnosticar y resolver problemas técnicos.",
    backstory="Ingeniero de soporte nivel 2.",
    tools=[tool_kb, tool_incidencias, tool_pedido],
    llm=llm, verbose=True, allow_delegation=False
)

agente_consultas = Agent(
    role="Especialista en Consultas Generales",
    goal="Responder dudas sobre pedidos y políticas.",
    backstory="Agente de atención omnicanal.",
    tools=[tool_pedido, tool_cliente, tool_incidencias],
    llm=llm, verbose=True, allow_delegation=False
)

agente_qa = Agent(
    role="Agente de Control de Calidad",
    goal="Revisar y aprobar respuestas finales.",
    backstory="Guardián de la excelencia en el servicio.",
    tools=[],
    llm=llm, verbose=True, allow_delegation=False
)

console.print("[bold green]✅ Agentes re-vinculados a las nuevas herramientas corregidas[/bold green]")

✅ Agentes re-vinculados a las nuevas herramientas corregidas

## 6. Definición de tareas y construcción del Crew

Las tareas se crean dinámicamente para cada ticket, encadenando los agentes
en el flujo correcto según la categoría detectada por el triaje.


In [ ]:
def construir_crew(ticket: dict) -> Crew:
    """
    Construye un Crew secuencial para procesar un ticket específico.
    El contexto del cliente y pedido se inyecta en la primera tarea.
    """
    cliente = ticket.get("cliente", {})
    pedido  = ticket.get("pedido", {})
    segmento = cliente.get("segmento", "estandar").upper()

    contexto_base = f"""
TICKET: {ticket['id']}
SEGMENTO CLIENTE: {segmento}
CLIENTE ID: {ticket.get('cliente_id', 'desconocido')}
PEDIDO ID: {ticket.get('pedido_id', 'desconocido')}
PRODUCTO: {pedido.get('producto', {}).get('nombre', 'desconocido')}
DÍAS DESDE COMPRA: {pedido.get('dias_desde_compra', '?')}

MENSAJE DEL CLIENTE:
{ticket['mensaje']}
"""

    # ── Tarea 1: Triaje ───────────────────────────────────────────────────────
    tarea_triage = Task(
        description=f"""
Analiza el siguiente ticket de soporte y clasifícalo.

{contexto_base}

Devuelve un JSON con esta estructura exacta:
{{
  "categoria": "devolucion|soporte_tecnico|consulta|queja_grave",
  "prioridad": "alta|media|baja",
  "resumen": "máximo 20 palabras describiendo el problema",
  "idioma": "es|en|fr",
  "requiere_escalado_inmediato": true/false,
  "razon_escalado": "solo si requiere_escalado_inmediato=true"
}}

Criterios de prioridad:
- alta: cliente VIP, queja grave, bloqueo total del producto, amenaza legal
- media: problema funcional parcial, devolución con complicaciones
- baja: consulta informativa, devolución sencilla
""",
        expected_output="JSON válido con la clasificación del ticket.",
        agent=agente_triage,
    )

    # ── Tarea 2: Resolución especializada ─────────────────────────────────────
    categoria = ticket.get("categoria", "consulta")

    if categoria == "devolucion":
        agente_resolucion = agente_devoluciones
        descripcion_resolucion = f"""
Gestiona la solicitud de devolución para el ticket {ticket['id']}.

{contexto_base}

Pasos obligatorios:
1. Usa 'consultar_pedido' para obtener los detalles completos del pedido.
2. Usa 'verificar_devolucion' para comprobar la elegibilidad.
3. Si es elegible, usa 'crear_etiqueta_devolucion' para generar la etiqueta.
4. Redacta una respuesta empática con instrucciones claras paso a paso.

Si el cliente es VIP (segmento=vip), ofrece recogida en domicilio sin coste adicional.
Menciona siempre el plazo de reembolso.
"""
    elif categoria == "soporte_tecnico":
        agente_resolucion = agente_soporte
        descripcion_resolucion = f"""
Resuelve el problema técnico del ticket {ticket['id']}.

{contexto_base}

Pasos obligatorios:
1. Usa 'verificar_incidencias' para comprobar si hay problemas conocidos.
2. Usa 'buscar_knowledge_base' con el nombre del producto y el problema.
3. Redacta una solución paso a paso, clara y sin jerga técnica.

Si hay una incidencia activa que explica el problema, infórmalo antes de los pasos.
Si la KB no tiene solución, ofrece reemplazo por garantía si aplica.
"""
    elif categoria == "queja_grave":
        agente_resolucion = agente_consultas
        descripcion_resolucion = f"""
Gestiona esta queja grave con máxima prioridad — ticket {ticket['id']}.

{contexto_base}

El cliente está muy frustrado. Tu respuesta debe:
1. Disculparse genuinamente (sin excusas genéricas).
2. Reconocer exactamente los fallos que describe el cliente.
3. Proponer una solución concreta e inmediata (no promesas vagas).
4. Ofrecer compensación si el segmento es premium o vip.
5. Indicar que un responsable de cuenta le contactará en <2h.

Usa 'consultar_cliente' para conocer su historial y personalizar la respuesta.
"""
    else:  # consulta
        agente_resolucion = agente_consultas
        descripcion_resolucion = f"""
Responde la consulta del ticket {ticket['id']}.

{contexto_base}

Pasos:
1. Usa 'consultar_pedido' si la pregunta es sobre estado de envío o entrega.
2. Usa 'verificar_incidencias' si hay retrasos de los que informar.
3. Responde de forma directa, amigable y completa.

Si el cliente pregunta por el número de seguimiento, proporciona instrucciones
exactas para rastrearlo en la web del transportista.
"""

    tarea_resolucion = Task(
        description=descripcion_resolucion,
        expected_output=(
            "Borrador de respuesta profesional, empática y completa para el cliente. "
            "Incluir todos los datos relevantes (números de etiqueta, plazos, pasos)."
        ),
        agent=agente_resolucion,
        context=[tarea_triage],
    )

    # ── Tarea 3: QA ───────────────────────────────────────────────────────────
    tarea_qa = Task(
        description=f"""
Revisa el borrador de respuesta para el ticket {ticket['id']}.

MENSAJE ORIGINAL DEL CLIENTE:
{ticket['mensaje']}

Evalúa el borrador en 4 dimensiones (1-10 cada una):
1. PRECISIÓN: ¿Los hechos y datos son correctos?
2. POLÍTICA: ¿Cumple las políticas de la empresa?
3. TONO: ¿Es empático, claro y apropiado para el segmento {segmento}?
4. COMPLETITUD: ¿Responde todo lo que el cliente preguntó?

Produce tu evaluación en este formato:

## EVALUACIÓN QA
**Score global:** X/10

| Dimensión | Score | Observación |
|-----------|-------|-------------|
| Precisión | X/10 | ... |
| Política  | X/10 | ... |
| Tono      | X/10 | ... |
| Completitud | X/10 | ... |

**Decisión:** APROBADO / RECHAZADO / ESCALAR A HUMANO

**Respuesta final para el cliente:**
[Si APROBADO: copia la respuesta con pequeños ajustes si los hubiera]
[Si RECHAZADO: indica qué debe corregir el especialista]
[Si ESCALAR: indica por qué y qué información pasar al agente humano]
""",
        expected_output=(
            "Evaluación estructurada con scores, decisión final y respuesta aprobada "
            "o instrucciones de corrección."
        ),
        agent=agente_qa,
        context=[tarea_triage, tarea_resolucion],
    )

    return Crew(
        agents=[agente_triage, agente_resolucion, agente_qa],
        tasks=[tarea_triage, tarea_resolucion, tarea_qa],
        process=Process.sequential,
        verbose=True,
    )

console.print("[bold green]✅ Constructor de Crew listo[/bold green]")


✅ Constructor de Crew listo

## 7. Ejecución — Procesamiento de tickets

Procesamos los 5 tickets de muestra y mostramos los resultados de forma legible.


In [7]:
# ── Función para procesar un ticket individual ────────────────────────────────
def procesar_ticket(ticket):
    console.print(f"[bold blue]Procesando Ticket: {ticket['id']}...[/bold blue]")
    crew = construir_crew(ticket)
    resultado = crew.kickoff()
    return {
        "ticket_id": ticket['id'],
        "categoria": ticket['categoria'],
        "cliente": ticket['cliente']['nombre'],
        "segmento": ticket['cliente']['segmento'],
        "resultado": str(resultado)
    }

# ── Re-procesar todos los tickets ─────────────────────────────────────────────
# Nota: Asegúrate de haber ejecutado las celdas de 'Configuración' y 'Agentes'
# después de actualizar la API KEY para que los cambios surtan efecto.
resultados = []
for ticket in TICKETS_MUESTRA:
    try:
        res = procesar_ticket(ticket)
        resultados.append(res)
    except Exception as e:
        console.print(f"[bold red]Error en {ticket['id']}: {e}[/bold red]")
    console.print()

Procesando Ticket: TK-001...

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ea918b14-bccf-4e4e-987c-5d61d6699226                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Analiza el siguiente ticket de soporte y clasifícalo.                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  TICKET: TK-001                                                                                                 │
│  SEGMENTO CLIENTE: PREMIUM                                                                                      │
│  CLIENTE ID: C8520                                                                                              │
│  PEDIDO ID: ORD-860165                                                                                          │
│  PRODUCTO: SmartWatch Pro X2                                                                                    │
│  DÍAS DESDE COMPRA: 21                                                                                          │
│                                                                                                                 │
│  MENSAJE DEL CLIENTE:                                                                                           │
│  Hola, compré un SmartWatch Pro X2 hace 3 semanas y la pantalla ha dejado de funcionar tras una caída. El       │
│  reloj cayó desde mi mesita de noche, unos 70cm. ¿Cubre la garantía este tipo de daño? Si no, ¿puedo            │
│  devolverlo igualmente?                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  Devuelve un JSON con esta estructura exacta:                                                                   │
│  {                                                                                                              │
│    "categoria": "devolucion|soporte_tecnico|consulta|queja_grave",                                              │
│    "prioridad": "alta|media|baja",                                                                              │
│    "resumen": "máximo 20 palabras describiendo el problema",                                                    │
│    "idioma": "es|en|fr",                                                                                        │
│    "requiere_escalado_inmediato": true/false,                                                                   │
│    "razon_escalado": "solo si requiere_escalado_inmediato=true"                                                 │
│  }                                                                                                              │
│                                                                                                                 │
│  Criterios de prioridad:                                                                                        │
│  - alta: cliente VIP, queja grave, bloqueo total del producto, amenaza legal                                    │
│  - media: problema funcional parcial, devolución con complicaciones                                             │
│  - baja: consulta informativa, devolución sencilla                                                              │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente de Triaje y Clasificación                                                                        │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Analiza el siguiente ticket de soporte y clasifícalo.                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  TICKET: TK-001                                                                                                 │
│  SEGMENTO CLIENTE: PREMIUM                                                                                      │
│  CLIENTE ID: C8520                                                                                              │
│  PEDIDO ID: ORD-860165                                                                                          │
│  PRODUCTO: SmartWatch Pro X2                                                                                    │
│  DÍAS DESDE COMPRA: 21                                                                                          │
│                                                                                                                 │
│  MENSAJE DEL CLIENTE:                                                                                           │
│  Hola, compré un SmartWatch Pro X2 hace 3 semanas y la pantalla ha dejado de funcionar tras una caída. El       │
│  reloj cayó desde mi mesita de noche, unos 70cm. ¿Cubre la garantía este tipo de daño? Si no, ¿puedo            │
│  devolverlo igualmente?                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  Devuelve un JSON con esta estructura exacta:                                                                   │
│  {                                                                                                              │
│    "categoria": "devolucion|soporte_tecnico|consulta|queja_grave",                                              │
│    "prioridad": "alta|media|baja",                                                                              │
│    "resumen": "máximo 20 palabras describiendo el problema",                                                    │
│    "idioma": "es|en|fr",                                                                                        │
│    "requiere_escalado_inmediato": true/false,                                                                   │
│    "razon_escalado": "solo si requiere_escalado_inmediato=true"                                                 │
│  }                                                                                                              │
│                                                                                                                 │
│  Criterios de prioridad:                                                                                        │
│  - alta: cliente VIP, queja grave, bloqueo total del producto, amenaza legal                                    │
│  - media: problema funcional parcial, devolución con complicaciones                                             │
│  - baja: consulta informativa, devolución sencilla     

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: consultar_cliente                                                                                        │
│  Args: {'cliente_id': 'C8520'}                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente de Control de Calidad                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Revisa el borrador de respuesta para el ticket TK-005.                                                         │
│                                                                                                                 │
│  MENSAJE ORIGINAL DEL CLIENTE:                                                                                  │
│  Hola! Recibí la mochila TrailMaster hace una semana, está perfecta, pero me regalaron la misma y tengo dos     │
│  iguales. ¿Puedo devolverla sin problema? No tiene ningún defecto, ni siquiera la he usado. Gracias 😊          │
│                                                                                                                 │
│  Evalúa el borrador en 4 dimensiones (1-10 cada una):                                                           │
│  1. PRECISIÓN: ¿Los hechos y datos son correctos?                                                               │
│  2. POLÍTICA: ¿Cumple las políticas de la empresa?                                                              │
│  3. TONO: ¿Es empático, claro y apropiado para el segmento ESTANDAR?                                            │
│  4. COMPLETITUD: ¿Responde todo lo que el cliente preguntó?                                                     │
│                                                                                                                 │
│  Produce tu evaluación en este formato:                                                                         │
│                                                                                                                 │
│  ## EVALUACIÓN QA                                                                                               │
│  **Score global:** X/10                                                                                         │
│                                                                                                                 │
│  | Dimensión | Score | Observación |                                                                            │
│  |-----------|-------|-------------|                                                                            │
│  | Precisión | X/10 | ... |                                                                                     │
│  | Política  | X/10 | ... |                                                                                     │
│  | Tono      | X/10 | ... |                                                                                     │
│  | Completitud | X/10 | ... |                                                                                   │
│                                                                                                                 │
│  **Decisión:** APROBADO / RECHAZADO / ESCALAR A HUMANO                                                          │
│                                                                                                                 │
│  **Respuesta final para el cliente:**                                                                           │
│  [Si APROBADO: copia la respuesta con pequeños ajustes si los hubiera]                                          │
│  [Si RECHAZADO: indica qué debe corregir el especialista]                                                       │
│  [Si ESCALAR: indica por qué y qué información pasar al 

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente de Control de Calidad                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## EVALUACIÓN QA                                                                                               │
│  **Score global:** 8/10                                                                                         │
│                                                                                                                 │
│  | Dimensión | Score | Observación |                                                                            │
│  |-----------|-------|-------------|                                                                            │
│  | Precisión | 9/10 | La información sobre la devolución es correcta, pero el número de etiqueta debe ser       │
│  verificado para asegurar que sea válido. |                                                                     │
│  | Política  | 8/10 | Cumple con la política de devoluciones, pero sería útil mencionar que el cliente debe     │
│  incluir el recibo o comprobante de compra en el paquete de devolución. |                                       │
│  | Tono      | 9/10 | El tono es empático y amigable, adecuado para el segmento estándar. |                     │
│  | Completitud | 7/10 | Responde a la pregunta del cliente, pero falta información sobre la necesidad de        │
│  incluir el recibo o comprobante de compra. |                                                                   │
│                                                                                                                 │
│  **Decisión:** RECHAZADO                                                                                        │
│                                                                                                                 │
│  **Respuesta final para el cliente:**                                                                           │
│  Estimado cliente,                                                                                              │
│                                                                                                                 │
│  ¡Gracias por contactarnos! Entendemos que deseas devolver la mochila TrailMaster 40L que recibiste hace una    │
│  semana. Nos alegra informarte que tu solicitud de devolución ha sido aprobada, ya que estás dentro del plazo   │
│  de 30 días y el producto no presenta defectos.                                                                 │
│                                                                                                                 │
│  Aquí tienes los pasos a seguir para completar la devolución:                                                   │
│                                                                                                                 │
│  1. **Etiqueta de Devolución**: Hemos generado una etiqueta de devolución para ti. El número de etiqueta es     │
│  **RET-9397-QDZD**.                                                                                             │
│  2. **Transportista**: Debes llevar el paquete a cualquier oficina de **SEUR** en los próximos **14 días**.     │
│  3. **Reembolso**: Una vez que recibamos el paquete, procesaremos tu reembolso, que se realizará en un plazo    │
│  de **3 a 5 días hábiles**.                                                                                     │
│  4. **Comprobante de Compra**: Por favor, asegúrate de 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Revisa el borrador de respuesta para el ticket TK-005.                                                         │
│                                                                                                                 │
│  MENSAJE ORIGINAL DEL CLIENTE:                                                                                  │
│  Hola! Recibí la mochila TrailMaster hace una semana, está perfecta, pero me regalaron la misma y tengo dos     │
│  iguales. ¿Puedo devolverla sin problema? No tiene ningún defecto, ni siquiera la he usado. Gracias 😊          │
│                                                                                                                 │
│  Evalúa el borrador en 4 dimensiones (1-10 cada una):                                                           │
│  1. PRECISIÓN: ¿Los hechos y datos son correctos?                                                               │
│  2. POLÍTICA: ¿Cumple las políticas de la empresa?                                                              │
│  3. TONO: ¿Es empático, claro y apropiado para el segmento ESTANDAR?                                            │
│  4. COMPLETITUD: ¿Responde todo lo que el cliente preguntó?                                                     │
│                                                                                                                 │
│  Produce tu evaluación en este formato:                                                                         │
│                                                                                                                 │
│  ## EVALUACIÓN QA                                                                                               │
│  **Score global:** X/10                                                                                         │
│                                                                                                                 │
│  | Dimensión | Score | Observación |                                                                            │
│  |-----------|-------|-------------|                                                                            │
│  | Precisión | X/10 | ... |                                                                                     │
│  | Política  | X/10 | ... |                                                                                     │
│  | Tono      | X/10 | ... |                                                                                     │
│  | Completitud | X/10 | ... |                                                                                   │
│                                                                                                                 │
│  **Decisión:** APROBADO / RECHAZADO / ESCALAR A HUMANO                                                          │
│                                                                                                                 │
│  **Respuesta final para el cliente:**                                                                           │
│  [Si APROBADO: copia la respuesta con pequeños ajustes si los hubiera]                                          │
│  [Si RECHAZADO: indica qué debe corregir el especialista]                                                       │
│  [Si ESCALAR: indica por qué y qué información pasar al agente humano]                                          │
│                                                         

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 98259922-c4b5-4450-956f-239bee51ffb7                                                                       │
│  Final Output: ## EVALUACIÓN QA                                                                                 │
│  **Score global:** 8/10                                                                                         │
│                                                                                                                 │
│  | Dimensión | Score | Observación |                                                                            │
│  |-----------|-------|-------------|                                                                            │
│  | Precisión | 9/10 | La información sobre la devolución es correcta, pero el número de etiqueta debe ser       │
│  verificado para asegurar que sea válido. |                                                                     │
│  | Política  | 8/10 | Cumple con la política de devoluciones, pero sería útil mencionar que el cliente debe     │
│  incluir el recibo o comprobante de compra en el paquete de devolución. |                                       │
│  | Tono      | 9/10 | El tono es empático y amigable, adecuado para el segmento estándar. |                     │
│  | Completitud | 7/10 | Responde a la pregunta del cliente, pero falta información sobre la necesidad de        │
│  incluir el recibo o comprobante de compra. |                                                                   │
│                                                                                                                 │
│  **Decisión:** RECHAZADO                                                                                        │
│                                                                                                                 │
│  **Respuesta final para el cliente:**                                                                           │
│  Estimado cliente,                                                                                              │
│                                                                                                                 │
│  ¡Gracias por contactarnos! Entendemos que deseas devolver la mochila TrailMaster 40L que recibiste hace una    │
│  semana. Nos alegra informarte que tu solicitud de devolución ha sido aprobada, ya que estás dentro del plazo   │
│  de 30 días y el producto no presenta defectos.                                                                 │
│                                                                                                                 │
│  Aquí tienes los pasos a seguir para completar la devolución:                                                   │
│                                                                                                                 │
│  1. **Etiqueta de Devolución**: Hemos generado una etiqueta de devolución para ti. El número de etiqueta es     │
│  **RET-9397-QDZD**.                                                                                             │
│  2. **Transportista**: Debes llevar el paquete a cualquier oficina de **SEUR** en los próximos **14 días**.     │
│  3. **Reembolso**: Una vez que recibamos el paquete, procesaremos tu reembolso, que se realizará en un plazo    │
│  de **3 a 5 días hábiles**.                                                                                     │
│  4. **Comprobante de Compra**: Por favor, asegúrate de

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 8. Resumen de resultados

In [8]:
tabla_res = Table(
    title="📋 Resumen de Tickets Procesados",
    show_header=True,
    header_style="bold magenta",
)
tabla_res.add_column("Ticket",    style="bold")
tabla_res.add_column("Categoría", style="cyan")
tabla_res.add_column("Cliente")
tabla_res.add_column("Segmento")
tabla_res.add_column("Estado",    style="green")

for r in resultados:
    tabla_res.add_row(
        r["ticket_id"],
        r["categoria"],
        r["cliente"],
        r["segmento"],
        "✅ Procesado",
    )

console.print(tabla_res)
console.print(f"\n[bold green]✅ {len(resultados)} tickets procesados correctamente.[/bold green]")


                        📋 Resumen de Tickets Procesados                        
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Ticket ┃ Categoría       ┃ Cliente                 ┃ Segmento ┃ Estado       ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ TK-001 │ devolucion      │ Saturnino Alcázar Armas │ premium  │ ✅ Procesado │
│ TK-002 │ soporte_tecnico │ Américo Carpio Porcel   │ estandar │ ✅ Procesado │
│ TK-003 │ consulta        │ Adelina Hernando        │ premium  │ ✅ Procesado │
│ TK-004 │ queja_grave     │ Ramona Macías Oliver    │ estandar │ ✅ Procesado │
│ TK-005 │ devolucion      │ Belen Carreras Villegas │ estandar │ ✅ Procesado │
└────────┴─────────────────┴─────────────────────────┴──────────┴──────────────┘

✅ 5 tickets procesados correctamente.

## 9. Prueba con ticket personalizado

Escribe tu propio mensaje de cliente para ver cómo lo procesa el sistema.


In [9]:
# 👇 Modifica este mensaje y vuelve a ejecutar la celda
MENSAJE_PERSONALIZADO = """
Hola, recibí mis auriculares NoiseX hace 5 días pero el micrófono
no funciona cuando hago llamadas. En Spotify el sonido es perfecto,
pero en WhatsApp y Teams la gente dice que no me escucha nada.
Ya reinicié el teléfono y nada. ¿Qué puedo hacer?
"""

# ── Buscamos dinámicamente un pedido que tenga los auriculares NoiseX ──
pedidos_noisex = [k for k, v in PEDIDOS.items() if 'NoiseX' in v['producto']['nombre']]
pedido_id_dinamico = pedidos_noisex[0] if pedidos_noisex else list(PEDIDOS.keys())[0]
cliente_id_dinamico = PEDIDOS[pedido_id_dinamico]['cliente_id']

ticket_custom = {
    "id":        "TK-CUSTOM",
    "categoria": "soporte_tecnico",   # puede cambiarse: devolucion|consulta|queja_grave
    "mensaje":   MENSAJE_PERSONALIZADO.strip(),
    "cliente_id": cliente_id_dinamico,
    "pedido_id":  pedido_id_dinamico,
    "cliente":    CLIENTES[cliente_id_dinamico],
    "pedido":     PEDIDOS[pedido_id_dinamico],
}

resultado_custom = procesar_ticket(ticket_custom)
console.print(Panel(resultado_custom["resultado"], title="✅ Respuesta final aprobada por QA", style="green"))



Procesando Ticket: TK-CUSTOM...

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b7caa1d0-d4f6-4374-ac2a-4d1c8ee56d9a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Analiza el siguiente ticket de soporte y clasifícalo.                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  TICKET: TK-CUSTOM                                                                                              │
│  SEGMENTO CLIENTE: ESTANDAR                                                                                     │
│  CLIENTE ID: C1512                                                                                              │
│  PEDIDO ID: ORD-861550                                                                                          │
│  PRODUCTO: Auriculares BT NoiseX                                                                                │
│  DÍAS DESDE COMPRA: 31                                                                                          │
│                                                                                                                 │
│  MENSAJE DEL CLIENTE:                                                                                           │
│  Hola, recibí mis auriculares NoiseX hace 5 días pero el micrófono                                              │
│  no funciona cuando hago llamadas. En Spotify el sonido es perfecto,                                            │
│  pero en WhatsApp y Teams la gente dice que no me escucha nada.                                                 │
│  Ya reinicié el teléfono y nada. ¿Qué puedo hacer?                                                              │
│                                                                                                                 │
│                                                                                                                 │
│  Devuelve un JSON con esta estructura exacta:                                                                   │
│  {                                                                                                              │
│    "categoria": "devolucion|soporte_tecnico|consulta|queja_grave",                                              │
│    "prioridad": "alta|media|baja",                                                                              │
│    "resumen": "máximo 20 palabras describiendo el problema",                                                    │
│    "idioma": "es|en|fr",                                                                                        │
│    "requiere_escalado_inmediato": true/false,                                                                   │
│    "razon_escalado": "solo si requiere_escalado_inmediato=true"                                                 │
│  }                                                                                                              │
│                                                                                                                 │
│  Criterios de prioridad:                                                                                        │
│  - alta: cliente VIP, queja grave, bloqueo total del producto, amenaza legal                                    │
│  - media: problema funcional parcial, devolución con complicaciones                                             │
│  - baja: consulta informativa, devolución sencilla     

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente de Triaje y Clasificación                                                                        │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Analiza el siguiente ticket de soporte y clasifícalo.                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  TICKET: TK-CUSTOM                                                                                              │
│  SEGMENTO CLIENTE: ESTANDAR                                                                                     │
│  CLIENTE ID: C1512                                                                                              │
│  PEDIDO ID: ORD-861550                                                                                          │
│  PRODUCTO: Auriculares BT NoiseX                                                                                │
│  DÍAS DESDE COMPRA: 31                                                                                          │
│                                                                                                                 │
│  MENSAJE DEL CLIENTE:                                                                                           │
│  Hola, recibí mis auriculares NoiseX hace 5 días pero el micrófono                                              │
│  no funciona cuando hago llamadas. En Spotify el sonido es perfecto,                                            │
│  pero en WhatsApp y Teams la gente dice que no me escucha nada.                                                 │
│  Ya reinicié el teléfono y nada. ¿Qué puedo hacer?                                                              │
│                                                                                                                 │
│                                                                                                                 │
│  Devuelve un JSON con esta estructura exacta:                                                                   │
│  {                                                                                                              │
│    "categoria": "devolucion|soporte_tecnico|consulta|queja_grave",                                              │
│    "prioridad": "alta|media|baja",                                                                              │
│    "resumen": "máximo 20 palabras describiendo el problema",                                                    │
│    "idioma": "es|en|fr",                                                                                        │
│    "requiere_escalado_inmediato": true/false,                                                                   │
│    "razon_escalado": "solo si requiere_escalado_inmediato=true"                                                 │
│  }                                                                                                              │
│                                                                                                                 │
│  Criterios de prioridad:                                                                                        │
│  - alta: cliente VIP, queja grave, bloqueo total del producto, amenaza legal                                    │
│  - media: problema funcional parcial, devolución con co

Tool consultar_cliente executed with result: {
  "id": "C1512",
  "nombre": "Ramona Macías Oliver",
  "email": "arcosabina@example.com",
  "telefono": "+34 948 33 37 37",
  "segmento": "estandar",
  "fecha_registro": "2025-03-16",
  "total_pedid...

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente de Control de Calidad                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## EVALUACIÓN QA                                                                                               │
│  **Score global:** 8/10                                                                                         │
│                                                                                                                 │
│  | Dimensión | Score | Observación |                                                                            │
│  |-----------|-------|-------------|                                                                            │
│  | Precisión | 9/10 | La información sobre el emparejamiento y los pasos para reiniciar los auriculares es      │
│  correcta. Sin embargo, sería útil mencionar que el problema podría estar relacionado con la configuración de   │
│  permisos de las aplicaciones. |                                                                                │
│  | Política  | 10/10 | La respuesta cumple con las políticas de la empresa, incluyendo la oferta de un          │
│  reemplazo por garantía. |                                                                                      │
│  | Tono      | 8/10 | El tono es mayormente empático y claro, aunque podría ser un poco más cálido en la        │
│  apertura y cierre. |                                                                                           │
│  | Completitud | 7/10 | Responde a la mayoría de las inquietudes del cliente, pero no aborda la posibilidad de  │
│  problemas de configuración de permisos en las aplicaciones. |                                                  │
│                                                                                                                 │
│  **Decisión:** APROBADO                                                                                         │
│                                                                                                                 │
│  **Respuesta final para el cliente:**                                                                           │
│  Estimado cliente,                                                                                              │
│                                                                                                                 │
│  Gracias por contactarnos y lamento los inconvenientes que está experimentando con sus auriculares BT NoiseX.   │
│  He revisado su caso y aquí están los pasos que puede seguir para intentar resolver el problema del micrófono:  │
│                                                                                                                 │
│  1. **Reinicie los auriculares**: Para resetear los auriculares, toque ambas orejeras tres veces seguidas.      │
│  Esto puede ayudar a restablecer la conexión del micrófono.                                                     │
│                                                                                                                 │
│  2. **Verifique el emparejamiento**: Asegúrese de que los auriculares estén correctamente emparejados con su    │
│  dispositivo. Si es necesario, apague los auriculares y mantenga presionado el botón durante 5 segundos hasta   │
│  que el LED parpadee, lo que indica que están listos para emparejarse nuevamente.                               │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Revisa el borrador de respuesta para el ticket TK-CUSTOM.                                                      │
│                                                                                                                 │
│  MENSAJE ORIGINAL DEL CLIENTE:                                                                                  │
│  Hola, recibí mis auriculares NoiseX hace 5 días pero el micrófono                                              │
│  no funciona cuando hago llamadas. En Spotify el sonido es perfecto,                                            │
│  pero en WhatsApp y Teams la gente dice que no me escucha nada.                                                 │
│  Ya reinicié el teléfono y nada. ¿Qué puedo hacer?                                                              │
│                                                                                                                 │
│  Evalúa el borrador en 4 dimensiones (1-10 cada una):                                                           │
│  1. PRECISIÓN: ¿Los hechos y datos son correctos?                                                               │
│  2. POLÍTICA: ¿Cumple las políticas de la empresa?                                                              │
│  3. TONO: ¿Es empático, claro y apropiado para el segmento ESTANDAR?                                            │
│  4. COMPLETITUD: ¿Responde todo lo que el cliente preguntó?                                                     │
│                                                                                                                 │
│  Produce tu evaluación en este formato:                                                                         │
│                                                                                                                 │
│  ## EVALUACIÓN QA                                                                                               │
│  **Score global:** X/10                                                                                         │
│                                                                                                                 │
│  | Dimensión | Score | Observación |                                                                            │
│  |-----------|-------|-------------|                                                                            │
│  | Precisión | X/10 | ... |                                                                                     │
│  | Política  | X/10 | ... |                                                                                     │
│  | Tono      | X/10 | ... |                                                                                     │
│  | Completitud | X/10 | ... |                                                                                   │
│                                                                                                                 │
│  **Decisión:** APROBADO / RECHAZADO / ESCALAR A HUMANO                                                          │
│                                                                                                                 │
│  **Respuesta final para el cliente:**                                                                           │
│  [Si APROBADO: copia la respuesta con pequeños ajustes si los hubiera]                                          │
│  [Si RECHAZADO: indica qué debe corregir el especialist

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b7caa1d0-d4f6-4374-ac2a-4d1c8ee56d9a                                                                       │
│  Final Output: ## EVALUACIÓN QA                                                                                 │
│  **Score global:** 8/10                                                                                         │
│                                                                                                                 │
│  | Dimensión | Score | Observación |                                                                            │
│  |-----------|-------|-------------|                                                                            │
│  | Precisión | 9/10 | La información sobre el emparejamiento y los pasos para reiniciar los auriculares es      │
│  correcta. Sin embargo, sería útil mencionar que el problema podría estar relacionado con la configuración de   │
│  permisos de las aplicaciones. |                                                                                │
│  | Política  | 10/10 | La respuesta cumple con las políticas de la empresa, incluyendo la oferta de un          │
│  reemplazo por garantía. |                                                                                      │
│  | Tono      | 8/10 | El tono es mayormente empático y claro, aunque podría ser un poco más cálido en la        │
│  apertura y cierre. |                                                                                           │
│  | Completitud | 7/10 | Responde a la mayoría de las inquietudes del cliente, pero no aborda la posibilidad de  │
│  problemas de configuración de permisos en las aplicaciones. |                                                  │
│                                                                                                                 │
│  **Decisión:** APROBADO                                                                                         │
│                                                                                                                 │
│  **Respuesta final para el cliente:**                                                                           │
│  Estimado cliente,                                                                                              │
│                                                                                                                 │
│  Gracias por contactarnos y lamento los inconvenientes que está experimentando con sus auriculares BT NoiseX.   │
│  He revisado su caso y aquí están los pasos que puede seguir para intentar resolver el problema del micrófono:  │
│                                                                                                                 │
│  1. **Reinicie los auriculares**: Para resetear los auriculares, toque ambas orejeras tres veces seguidas.      │
│  Esto puede ayudar a restablecer la conexión del micrófono.                                                     │
│                                                                                                                 │
│  2. **Verifique el emparejamiento**: Asegúrese de que los auriculares estén correctamente emparejados con su    │
│  dispositivo. Si es necesario, apague los auriculares y mantenga presionado el botón durante 5 segundos hasta   │
│  que el LED parpadee, lo que indica que están listos para emparejarse nuevamente.                               │
│                                                       

╭────────────────────────────────────── ✅ Respuesta final aprobada por QA ───────────────────────────────────────╮
│ ## EVALUACIÓN QA                                                                                                │
│ **Score global:** 8/10                                                                                          │
│                                                                                                                 │
│ | Dimensión | Score | Observación |                                                                             │
│ |-----------|-------|-------------|                                                                             │
│ | Precisión | 9/10 | La información sobre el emparejamiento y los pasos para reiniciar los auriculares es       │
│ correcta. Sin embargo, sería útil mencionar que el problema podría estar relacionado con la configuración de    │
│ permisos de las aplicaciones. |                                                                                 │
│ | Política  | 10/10 | La respuesta cumple con las políticas de la empresa, incluyendo la oferta de un reemplazo │
│ por garantía. |                                                                                                 │
│ | Tono      | 8/10 | El tono es mayormente empático y claro, aunque podría ser un poco más cálido en la         │
│ apertura y cierre. |                                                                                            │
│ | Completitud | 7/10 | Responde a la mayoría de las inquietudes del cliente, pero no aborda la posibilidad de   │
│ problemas de configuración de permisos en las aplicaciones. |                                                   │
│                                                                                                                 │
│ **Decisión:** APROBADO                                                                                          │
│                                                                                                                 │
│ **Respuesta final para el cliente:**                                                                            │
│ Estimado cliente,                                                                                               │
│                                                                                                                 │
│ Gracias por contactarnos y lamento los inconvenientes que está experimentando con sus auriculares BT NoiseX. He │
│ revisado su caso y aquí están los pasos que puede seguir para intentar resolver el problema del micrófono:      │
│                                                                                                                 │
│ 1. **Reinicie los auriculares**: Para resetear los auriculares, toque ambas orejeras tres veces seguidas. Esto  │
│ puede ayudar a restablecer la conexión del micrófono.                                                           │
│                                                                                                                 │
│ 2. **Verifique el emparejamiento**: Asegúrese de que los auriculares estén correctamente emparejados con su     │
│ dispositivo. Si es necesario, apague los auriculares y mantenga presionado el botón durante 5 segundos hasta    │
│ que el LED parpadee, lo que indica que están listos para emparejarse nuevamente.                                │
│                                                                                                                 │
│ 3. **Pruebe en otro dispositivo**: Si es posible, pruebe los auriculares con otro teléfono o dispositivo para   │
│ ver si el problema persiste. Esto ayudará a determinar si el problema está en los auriculares o en el           │
│ dispositivo original.                                                                                           │
│                                                        

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯